# Creating Gold Tables

The purpose of this notebook is to create gold layer tables by performing business-level aggregations on silver layer tables.

The following steps were taken:
1. Loaded silver tables from previous step into DataFrames.
2. Created 7 gold tables by performing aggregations using joins, groupBy, sum, count, and window functions.
3. Wrote the aggregated results out to gold-layer Delta tables.

### Loading silver tables into DataFrames

In [0]:
customer_df = spark.read.format("delta").table("workspace.retail_schema.customers_silver")
products_df = spark.read.format("delta").table("workspace.retail_schema.products_silver")
store_df = spark.read.format("delta").table("workspace.retail_schema.store_silver")

### 1. Total sales by customer gold table

In [0]:
from pyspark.sql.functions import round, sum, countDistinct, count

sales_by_cust = store_df.join(customer_df, on="Customer_ID", how="inner").groupBy("Customer_ID", "Customer_Name", "Segment", "Region").agg(
    sum("Sales").alias("Total_Sales"),
    sum("Discount").alias("Total_Discount"),
    countDistinct("Order_ID").alias("Order_Count")
)

# Rounding the results
sales_by_cust = sales_by_cust.select(
    "Customer_ID", "Customer_Name", "Segment", "Region", round("Total_Sales", 2).alias("Total_Sales"),
    round("Total_Discount", 4).alias("Total_Discount"),
    "Order_Count" 
)

# Writing to the Gold Level Table
sales_by_cust.write.format("delta").mode("overwrite").saveAsTable("workspace.retail_schema.sales_by_customer_gold")

### 2. Total sales for each product gold tale

In [0]:
sales_by_prod = store_df.join(products_df, on="Product_ID", how="inner").groupBy("Product_ID", "Product_Name", "Category", "Sub_Category").agg(
    sum("Sales").alias("Total_Sales"),
    countDistinct("Order_ID").alias("Total_No_Of_Orders")
)

# Rounding the results
sales_by_prod = sales_by_prod.select(
    "Product_ID", "Product_Name", "Category", "Sub_Category", 
    round("Total_Sales", 2).alias("Total_Sales"),
    "Total_No_Of_Orders"
)

# Writing to the Gold Level Table
sales_by_prod.write.format("delta").mode("overwrite").saveAsTable("workspace.retail_schema.product_sales_gold")

### 3. Daily sales summary over the 4 years gold table

In [0]:
from pyspark.sql.functions import avg

daily_sales = store_df.groupBy("Order_Date").agg(
  sum("Sales").alias("Total_Sales"),
  avg("Discount").alias("Average_Discount"),
  avg("Shipping_Delivery_Days").alias("Average_Delivery_Days")
)

# Rounding the results
daily_sales = daily_sales.select(
    "Order_Date",
    round("Total_Sales", 2).alias("Total_Sales"),
    round("Average_Discount", 4).alias("Average_Discount"),
    round("Average_Delivery_Days", 2).alias("Average_Delivery_Days")
)

# Writing to the Gold level table
daily_sales.write.format("delta").mode("overwrite").saveAsTable("workspace.retail_schema.daily_sales_summary_gold")

### 4. Customer segment analysis by region gold table

In [0]:
customer_seg = store_df.join(customer_df, on="Customer_ID", how="inner").groupBy("Segment", "Region").agg(
  countDistinct("Customer_ID").alias("Total_Customers"),
  avg("Sales").alias("Average_Sales"),
  avg("Age").alias("Average_Age")
)

# Rounding the results
customer_seg = customer_seg.select(
  "Segment", "Region", "Total_Customers",
  round("Average_Sales", 2).alias("Average_Sales"),
  round("Average_Age", 2).alias("Average_Age")
)

# Writing to the Gold level table
customer_seg.write.format("delta").mode("overwrite").saveAsTable("workspace.retail_schema.customer_segmentation_region_gold")

### 5. Customer segmentation by city gold table

In [0]:
customer_seg_city = store_df.join(customer_df, on="Customer_ID", how="inner").groupBy("Segment", "City").agg(
  countDistinct("Customer_ID").alias("Total_Customers"),
  avg("Sales").alias("Average_Sales"),
  avg("Age").alias("Average_Age")
)

# Rounding the results
customer_seg_city = customer_seg_city.select(
  "Segment", "City", "Total_Customers",
  round("Average_Sales", 2).alias("Average_Sales"),
  round("Average_Age", 2).alias("Average_Age")
)

# Writing to the Gold level table
customer_seg_city.write.format("delta").mode("overwrite").saveAsTable("workspace.retail_schema.customer_segmentation_city_gold")

### 6. Customer-Product matrix gold table

In [0]:
cust_prod = store_df.groupBy("Customer_ID", "Product_ID").agg(
    sum("Sales").alias("Total_Sales"),
    count("Order_ID").alias("Purchase_Count"),
)

# Rounding the results
cust_prod = cust_prod.select(
    "Customer_ID", "Product_ID",
    round("Total_Sales", 2).alias("Total_Sales"),
    "Purchase_Count"
)

# Writing to the Gold level table
cust_prod.write.format("delta").mode("overwrite").partitionBy("Product_ID").saveAsTable("workspace.retail_schema.customer_product_matrix_gold")

### 7. Identifying the top `n` customers gold table

In [0]:
from pyspark.sql.functions import rank, col
from pyspark.sql.window import Window

dbutils.widgets.text("Top_n_customers", "50")
Top_n_customers = int(dbutils.widgets.get("Top_n_customers"))

top_cust = store_df.groupBy("Customer_ID").agg(
    sum("Sales").alias("Total_Sales"),
).withColumn("Rank", rank().over(Window.orderBy(col("Total_Sales").desc()))).filter(col("Rank") <= Top_n_customers)

# Rounding the results
top_cust = top_cust.select(
    "Customer_ID",
    round("Total_Sales", 2).alias("Total_Sales"),
    "Rank"
)

# Saving the Results in Gold Level Table
top_cust.write.format("delta").mode("overwrite").saveAsTable("workspace.retail_schema.top_customers_gold")